# Simple Python simulation smoke test

This notebook exercises the public Python API used by the examples: default `VERBOSITY=3` output columns, explicit verbosity selection, and forward source annotations.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Find build/ whether Jupyter is launched from the repository root or tests/smoke/.
cwd = Path.cwd()
repo_root = cwd.parents[1] if cwd.match("*/tests/smoke") else cwd
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens


## Default result

The default Python output uses the `VERBOSITY=3` style labels and includes relative proper-motion components.


In [ ]:
cfg = genulens.Config(l=1.0, b=-3.9, n_simu=2_000, seed=42)

result = genulens.simulate(cfg)
df = pd.DataFrame(result.to_numpy(), columns=result.columns)

required = [
    "wtj",
    "M_L",
    "D_L",
    "D_S",
    "t_E",
    "theta_E",
    "pi_E",
    "pi_EN",
    "pi_EE",
    "mu_rel",
    "mu_rel_N",
    "mu_rel_E",
    "mu_Sl",
    "mu_Sb",
    "I_L",
    "K_L",
    "iS",
    "iL",
    "fREM",
]
missing = [name for name in required if name not in df.columns]
assert not missing, missing
assert len(df) == cfg.n_simu
assert np.all(np.isfinite(df[required].to_numpy()))

df[["wtj", "M_L", "D_L", "D_S", "t_E", "theta_E", "pi_EN", "pi_EE", "mu_rel_N", "mu_rel_E"]].head()


## Explicit verbosity result

The default is already `VERBOSITY=3` style. This cell checks that another verbosity setting still returns its corresponding CLI-style labels.


In [ ]:
cfg_v1 = genulens.Config(l=1.0, b=-3.9, n_simu=1_000, seed=43)
cfg_v1.sampling.verbosity = 1

result_v1 = genulens.simulate(cfg_v1)
df_v1 = pd.DataFrame(result_v1.to_numpy(), columns=result_v1.columns)

expected = [
    "wtj",
    "tE",
    "thetaE",
    "piE",
    "M_L",
    "D_S",
    "D_L",
    "mu_rel",
    "iS",
    "iL",
    "tau_s",
    "tau_l",
    "fREM",
]
assert result_v1.columns == expected
assert np.all(np.isfinite(df_v1[["wtj", "M_L", "D_S", "D_L", "mu_rel"]].to_numpy()))

df_v1[expected].head()


## Forward source annotations

Forward source mode appends intrinsic source properties from the shared isochrone lookup. It is an annotation mode and does not replace the legacy LF/CMF event selection.


In [ ]:
cfg_src = genulens.Config(l=1.0, b=-3.9, n_simu=1_000, seed=44)
cfg_src.forward_source.enabled = 1
cfg_src.forward_source.photometry = "roman"
cfg_src.forward_source.min_initial_mass_msun = 0.1
cfg_src.forward_source.max_initial_mass_msun = 1.0

result_src = genulens.simulate(cfg_src)
df_src = pd.DataFrame(result_src.to_numpy(), columns=result_src.columns)

source_required = [
    "source_log_age",
    "source_metallicity_mh",
    "source_zini",
    "source_initial_mass_msun",
    "source_current_mass_msun",
    "source_radius_rsun",
    "source_teff_k",
    "source_logg",
    "source_angular_radius_microarcsec",
    "source_abs_F146mag",
]
missing = [name for name in source_required if name not in df_src.columns]
assert not missing, missing

finite_rows = np.isfinite(df_src[source_required].to_numpy()).all(axis=1)
assert finite_rows.mean() > 0.95

df_src.loc[finite_rows, source_required].head()
